In [1]:
import os
from pathlib import Path
from pypdf import PdfReader
import pandas as pd

# Define the path to raw PDF documents
DOCS_DIR = Path("../data/raw_docs")
pdf_files = sorted(list(DOCS_DIR.glob("*.pdf")))

print(f"Total PDF files detected: {len(pdf_files)}")

docs_summary = []
raw_documents = []

# Iterate through each PDF file in the directory
for file_path in pdf_files:
    reader = PdfReader(file_path)
    total_pages = len(reader.pages)
    failed_pages = 0
    full_text = ""
    
    # Extract text page by page
    for idx, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        # Normalize and remove redundant whitespaces
        cleaned_page_text = " ".join(text.split())
        
        # Check if the page is empty or requires OCR
        if len(cleaned_page_text.strip()) == 0:
            failed_pages += 1
        else:
            raw_documents.append({
                "source": file_path.name,
                "page": idx + 1,
                "content": cleaned_page_text
            })
            full_text += cleaned_page_text + " "
            
    # Append document inspection summary
    docs_summary.append({
        "Document": file_path.name,
        "Total Pages": total_pages,
        "Failed / Empty Pages": failed_pages,
        "Extracted Characters": len(full_text)
    })

# Render summary table as required by Phase 2.1
summary_df = pd.DataFrame(docs_summary)
display(summary_df)
print(f"\nTotal valid extracted page entries: {len(raw_documents)}")

Total PDF files detected: 4


Ignoring wrong pointing object 11 0 (offset 0)
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Subtype': '/Type1', '/BaseFont': '/RIMATZ+ApexNew-Bold', '/FontDescriptor': IndirectObject(26, 0, 1909382371792), '/Encoding': '/MacRomanEncoding', '/FirstChar': 32, '/LastChar': 119, '/Widths': [200, 0, 0, 0, 0, 0, 765, 0, 0, 0, 0, 0, 0, 0, 240, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1062, 640, 601, 562, 645, 560, 0, 641, 640, 274, 0, 645, 520, 792, 646, 680, 580, 0, 605, 595, 552, 640, 0, 890, 0, 656, 0, 0, 0, 0, 0, 0, 0, 498, 549, 440, 549, 509, 390, 540, 550, 275, 0, 0, 275, 803, 550, 520, 0, 0, 370, 470, 390, 550, 0, 690]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Subtype': '/Type1', '/BaseFont': '/MVSMHV+ApexNew-Book', '/FontDescriptor': IndirectObject(29, 0, 

,Document,Total Pages,Failed / Empty Pages,Extracted Characters
0,docker_overview.pdf,6,0,9556
1,git_cheat_sheet.pdf,2,0,4647
2,k8s_components.pdf,3,0,3530
3,linux_commands.pdf,12,0,17706



Total valid extracted page entries: 23


### 2.2 Chunking Strategy & Justification
- **Chunk Size:** 350 words (~450–500 tokens). This size preserves complete technical commands, syntax blocks, and descriptive paragraphs without diluting semantic meaning.
- **Chunk Overlap:** 60 words (~80 tokens). The overlap guarantees that contextual continuity between split sentences or command arguments is maintained across boundary cuts.
- **Strategy:** Sliding window chunking with metadata tracking (source document name and page number) to ensure verifiable, grounded citations.

In [2]:
# 2.2 Chunking Implementation
def chunk_text(text: str, chunk_size: int = 350, overlap: int = 60) -> list[str]:
    """
    Splits text into fixed-size word chunks with a sliding window overlap.
    """
    words = text.split()
    chunks = []
    start = 0
    
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        if end >= len(words):
            break
        start += (chunk_size - overlap)
        
    return chunks

# Generate chunks across all extracted pages
all_chunks = []
for doc in raw_documents:
    chunks = chunk_text(doc["content"], chunk_size=350, overlap=60)
    for c_idx, chunk_content in enumerate(chunks):
        all_chunks.append({
            "chunk_id": f"{doc['source']}_p{doc['page']}_c{c_idx}",
            "text": chunk_content,
            "source": doc["source"],
            "page": doc["page"]
        })

print(f"Total chunks created: {len(all_chunks)}")
if all_chunks:
    print("\n--- First Chunk Preview ---")
    print(f"ID: {all_chunks[0]['chunk_id']}")
    print(f"Source: {all_chunks[0]['source']} (Page {all_chunks[0]['page']})")
    print(f"Content: {all_chunks[0]['text'][:220]}...")

Total chunks created: 29

--- First Chunk Preview ---
ID: docker_overview.pdf_p1_c0
Source: docker_overview.pdf (Page 1)
Content: Home/ Get started/ What is Docker? What is Docker? Ask Gordon Copy Markdown View Markdown Docker is an open platform for developing, shipping, and running applications. Docker enables you to separate your applications fr...


In [ ]:
# 2.3 Embeddings Generation & ChromaDB Persistence
import chromadb
from sentence_transformers import SentenceTransformer

# Define persistent storage directory for ChromaDB
VECTOR_STORE_DIR = "../data/vector_store"
os.makedirs(VECTOR_STORE_DIR, exist_ok=True)

# Load lightweight and robust embedding model
embedding_model_name = "all-MiniLM-L6-v2"
embed_model = SentenceTransformer(embedding_model_name)

# Initialize persistent ChromaDB client
chroma_client = chromadb.PersistentClient(path=VECTOR_STORE_DIR)
collection = chroma_client.get_or_create_collection(
    name="rag_knowledge_base",
    metadata={"hnsw:space": "cosine"}
)

# Prepare payloads
chunk_ids = [c["chunk_id"] for c in all_chunks]
documents_text = [c["text"] for c in all_chunks]
metadatas = [{"source": c["source"], "page": c["page"]} for c in all_chunks]

# Compute dense vector embeddings
print(f"Generating embeddings using '{embedding_model_name}'...")
embeddings = embed_model.encode(documents_text, show_progress_bar=True).tolist()

# Upsert records into ChromaDB
collection.upsert(
    ids=chunk_ids,
    documents=documents_text,
    embeddings=embeddings,
    metadatas=metadatas
)

print(f"\nVector store successfully persisted to: {VECTOR_STORE_DIR}")
print(f"Total documents indexed in ChromaDB collection: {collection.count()}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\EL-Noby\miniconda3\envs\rag_env\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\EL-Noby\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings using 'all-MiniLM-L6-v2'...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Vector store successfully persisted to: ../data/vector_store
Total documents indexed in ChromaDB collection: 29


In [6]:
# 2.4 Refined Retrieval & Grounded Prompting
import ollama

def retrieve_context(query: str, n_results: int = 4) -> tuple[list[str], list[str]]:
    """
    Encodes the query and retrieves top-k relevant chunks from ChromaDB.
    """
    query_vector = embed_model.encode([query]).tolist()
    search_results = collection.query(
        query_embeddings=query_vector,
        n_results=n_results
    )
    
    contexts = []
    citations = []
    
    retrieved_docs = search_results["documents"][0]
    retrieved_metas = search_results["metadatas"][0]
    
    for doc, meta in zip(retrieved_docs, retrieved_metas):
        contexts.append(doc)
        citations.append(f"{meta['source']} (Page {meta['page']})")
        
    return contexts, list(dict.fromkeys(citations))

def ask_assistant(query: str, model_name: str = "llama3.2:1b") -> dict:
    """
    Retrieves relevant technical documentation and generates an accurate, grounded response.
    """
    contexts, citations = retrieve_context(query, n_results=4)
    combined_context = "\n\n".join(contexts)
    
    prompt = f"""You are a helpful technical assistant.
Answer the user question accurately using the information provided in the context below.
Be concise and factual. If the context truly does not contain relevant details to answer the question, reply with: "The provided documentation does not contain enough information to answer this question."

Context:
{combined_context}

User Question:
{query}

Answer:"""

    response = ollama.generate(
        model=model_name, 
        prompt=prompt,
        options={"temperature": 0.2}  # Low temperature for deterministic, grounded outputs
    )
    
    return {
        "query": query,
        "answer": response["response"].strip(),
        "citations": citations
    }

# Re-test technical query
test_query = "What is Docker and what does it enable developers to do?"
sample_output = ask_assistant(test_query)

print("Question:", sample_output["query"])
print("\nAnswer:\n", sample_output["answer"])
print("\nCitations / Sources:\n", sample_output["citations"])

Question: What is Docker and what does it enable developers to do?

Answer:
 Docker is an open platform for developing, shipping, and running applications. It enables developers to separate their applications from their infrastructure, allowing for fast, consistent delivery of applications. Docker streamlines the development lifecycle by allowing developers to work in standardized environments using local containers, which provide the necessary components to run the application. This makes it easier to develop, test, and deploy applications quickly and efficiently.

Citations / Sources:
 ['docker_overview.pdf (Page 1)', 'docker_overview.pdf (Page 3)', 'docker_overview.pdf (Page 2)']


In [7]:
# 2.6 Evaluation: Testing against 10 domain-specific technical queries
test_questions = [
    "What is Docker and what does it enable developers to do?",
    "What are the main components of the Kubernetes control plane?",
    "What is the function of the kube-apiserver in Kubernetes?",
    "Which command is used to initialize a new Git repository?",
    "How can you check the current status of tracked and untracked files in Git?",
    "What does the docker run command do?",
    "What is the role of the kube-scheduler component?",
    "Which Linux command is used to list directory contents?",
    "How do you create a new branch in Git?",
    "What is the difference between a container image and a container instance?"
]

evaluation_records = []

print("Running RAG evaluation across 10 sample queries...\n")
for q in test_questions:
    result = ask_assistant(q)
    
    # Check if context and citations were retrieved
    has_citations = len(result["citations"]) > 0
    not_empty = len(result["answer"]) > 15
    is_grounded = has_citations and not_empty and ("does not contain enough information" not in result["answer"].lower())
    
    evaluation_records.append({
        "Question": q,
        "Retrieved Sources": ", ".join(result["citations"]) if result["citations"] else "None",
        "Generated Answer": result["answer"][:120] + "...",
        "Grounded / Accurate": "Yes" if is_grounded else "Needs Review"
    })

eval_df = pd.DataFrame(evaluation_records)
display(eval_df)

Running RAG evaluation across 10 sample queries...



,Question,Retrieved Sources,Generated Answer,Grounded / Accurate
0,What is Docker and what does it enable develop...,"docker_overview.pdf (Page 1), docker_overview....","Docker is an open platform for developing, shi...",Yes
1,What are the main components of the Kubernetes...,"k8s_components.pdf (Page 1), k8s_components.pd...",The main components of the Kubernetes control ...,Yes
2,What is the function of the kube-apiserver in ...,"k8s_components.pdf (Page 1), k8s_components.pd...",The kube-apiserver is the core component serve...,Yes
3,Which command is used to initialize a new Git ...,"git_cheat_sheet.pdf (Page 1), git_cheat_sheet....",The command used to initialize a new Git repos...,Yes
4,How can you check the current status of tracke...,"git_cheat_sheet.pdf (Page 2), git_cheat_sheet....",To check the current status of tracked and unt...,Yes
5,What does the docker run command do?,"docker_overview.pdf (Page 5), docker_overview....",The `docker run` command is used to run a cont...,Yes
6,What is the role of the kube-scheduler component?,"k8s_components.pdf (Page 1), k8s_components.pd...",The kube-scheduler component is responsible fo...,Yes
7,Which Linux command is used to list directory ...,"linux_commands.pdf (Page 2), linux_commands.pd...",The Linux command used to list directory conte...,Yes
8,How do you create a new branch in Git?,"git_cheat_sheet.pdf (Page 2), git_cheat_sheet....","To create a new branch in Git, you can use the...",Yes
9,What is the difference between a container ima...,"docker_overview.pdf (Page 4), docker_overview....",A container image and a container instance are...,Yes


### 2.6 Failure Case Analysis & Mitigation Strategy
- **Observed Edge Cases:** 
  1. *Prompt Strictness*: Small LLMs (such as `llama3.2:1b`) can be overly cautious when asked to synthesize semantic paraphrases, resulting in premature fallback statements.
  2. *Chunk Boundary Truncation*: Questions requiring commands across consecutive lines risk separation if overlap is too small.
- **Mitigations Applied:**
  1. Adjusted generation temperature to `0.2` and refined the system prompt to guide factual extraction rather than exact word matching.
  2. Maintained a 60-word sliding overlap and retrieved top-4 chunks (`n_results=4`) to preserve complete command syntax and multi-component context.

In [11]:
# 2.7 Export Configuration for Backend Consumption
import json

config_payload = {
    "collection_name": "rag_knowledge_base",
    "embedding_model": "all-MiniLM-L6-v2",
    "llm_model": "llama3.2:1b",
    "n_results": 4,
    "temperature": 0.2
}

config_path = Path(VECTOR_STORE_DIR) / "rag_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config_payload, f, indent=4)

print(f"Configuration successfully exported to: {config_path.resolve()}")

Configuration successfully exported to: D:\AI_level_2_iti\rag-assistant-project\data\vector_store\rag_config.json
